StrOuputParser

In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate

In [11]:
load_dotenv()

True

In [12]:
model = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [13]:
template1 = PromptTemplate(
    template = "Write a detailed report on {topic}",
    input_variables = ["topic"]
)

In [14]:
template2 = PromptTemplate(
    template = "Write a 5-line summary on the following text. /n {text}",
    input_variables = ["text"]
)

In [15]:
parser = StrOutputParser()

In [16]:
chain = template1 | model | parser | template2 | model | parser

In [17]:
result = chain.invoke({"topic": "table tennis"})

In [18]:
print(result)

Table tennis, or ping pong, is a fast-paced racket sport originating in Victorian England and now played globally. It demands sharp reflexes, hand-eye coordination, and strategic thinking. The sport has evolved significantly with innovations like rubber paddles and became an Olympic sport in 1988. Beyond its competitive structure and physical benefits, table tennis offers significant mental and social advantages for players of all ages.


JSONOutputParser

In [19]:
from langchain_core.output_parsers import JsonOutputParser
parser = JsonOutputParser()

In [20]:
template3 = PromptTemplate(
    template = "Give me the name, age and city of a fictional person \n {format_instruction}",
    input_variables = [],
    partial_variables={"format_instruction": parser.get_format_instructions()}
)

In [21]:
prompt = template3.format()
result = model.invoke(prompt)
print(result)

content='```json\n{\n  "name": "Elara Vance",\n  "age": 29,\n  "city": "Oakhaven"\n}\n```' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019d196a-cfd0-7dd1-a891-345ccc925f62-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 20, 'output_tokens': 37, 'total_tokens': 57, 'input_token_details': {'cache_read': 0}}


In [22]:
parser.invoke(result)

{'name': 'Elara Vance', 'age': 29, 'city': 'Oakhaven'}

StructuredOutputParser

In [23]:
from langchain.output_parsers import StructuredOutputParser
from langchain import ResponseSchema

ModuleNotFoundError: No module named 'langchain.output_parsers'

In [ ]:
schema = [
    ResponseSchema(name="fact_1", description="Fact 1 about the topic"),
    ResponseSchema(name="fact_2", description="Fact 2 about the topic"),
    ResponseSchema(name="fact_3", description="Fact 3 about the topic")
]

parser = StructuredOutputParser.from_response_schemas(schema)

template = PromptTemplate(
    template = "Give 3 facts about {topic} \n {format_instruction}",
    input_variables=["topic"],
    partial_variables={"format_instruction": parser.get_format_instructions()}
)

prompt = template.invoke({"topic": "black hole"})
result = model.invoke(prompt)
parser.invoke(result)

NameError: name 'ResponseSchema' is not defined

PydanticOutputParser

In [24]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [25]:
class Person(BaseModel):
    name: str = Field(description="Name of the person")
    age: int = Field(gt=18, description="Age of the person")
    city: str = Field(description="Name of the city person belongs to")
    
parser = PydanticOutputParser(pydantic_object=Person)

template = PromptTemplate(
    template = "Give me the name, age and city of a fictional {place} person \n {format_instruction}",
    input_variables = ["place"],
    partial_variables={"format_instruction": parser.get_format_instructions()}
)

prompt = template.invoke({"place": "indian"})
result = model.invoke(prompt)
parser.invoke(result)

Person(name='Aarav Sharma', age=25, city='Jaipur')

In [26]:
#using chain
chain = template | model | parser
final_result = chain.invoke({"place": "korean"})
print(final_result)

name='Kim Ji-hoon' age=28 city='Seoul'
